In [3]:
import time
import json

from dotenv import load_dotenv
load_dotenv()
from app.data.dto.main.Coordinates import Coordinates

from app.services.utils import searoute_utils, email_sender
from app.data.dto.searoute.SearoutePath import SearoutePath
from app.data.dto.main.SeaPort import SeaPortDB, SeaPortDBIndexed
from app.services.db_service import DbService
from app.services.external_api.searoute_api import SearouteApi

sql_db_service = DbService()
await sql_db_service.init_pool()

searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

In [4]:
departure_port, err = await sql_db_service.get_port_by_locode("RULED")
destination_port, err = await sql_db_service.get_port_by_locode("AEJEA")
sea_route, err = await searoute_api.build_sea_route(departure_port.latitude, departure_port.longitude, destination_port.latitude, destination_port.longitude)

In [3]:
# closest_ports = []
# for route_area in sea_route.routeAreas:
#     time.sleep(0.5)
#     near_ports, err = await searoute_api.get_nearest_port_to_coordinates(route_area.latitude, route_area.longitude, {"limit": 5, "deviation": 100_000})
#     if not err:
#         for n_port in near_ports:
#            updated_port, err = await sql_db_service.upsert_port_size_from_searoute(n_port)
#         only_large_ports = [p for p in near_ports ]#if p.size == "large"]
#         closest_ports.extend(only_large_ports)
#
# unique_ports = searoute_utils.get_unique_ports(closest_ports)
# unique_ports.sort(key= lambda p: p.distance)
#
# ports_in_db = []
# for port in unique_ports:
#     port_db, err = await sql_db_service.get_port_by_locode(port.locode)
#     if not err:
#         ports_in_db.append(port_db)


In [4]:
#[p for p in unique_ports if p.locode == "ATTAM"]

In [5]:
# with open("./data/ports_mock.json", "w") as fp:
#     json.dump([p.model_dump() for p in ports_in_db], fp)
#
# with open("./data/coordinates.json", "w") as fp:
#     json.dump([c.model_dump() for c in sea_route.seaRouteCoordinates], fp)

In [5]:
from app.data import emoji
import json
from app.data.dto.main.Coordinates import Coordinates
from app.data.dto.main.SeaPort import SeaPortDB, SeaPortIndexed2

from map_drawing_app import worker

with open("./data/ports_mock.json", "r") as fp:
    ports = [SeaPortDB.from_db_row(p) for p in json.load(fp)]

with open("./data/coordinates.json", "r") as fp:
    coordinates = [Coordinates.from_dict(c) for c in json.load(fp)]


dep_port_indexed = departure_port.to_indexed2(index=emoji.ARROW_UP, color="green", size="medium", overlap=True)
dest_port_indexed = destination_port.to_indexed2(index=emoji.ARROW_DOWN, color="red", size="medium", overlap=True)

_set = set(list([departure_port.locode, destination_port.locode]))

other_ports = []
for port in ports:
    if port.locode not in _set:
        other_ports.append(port)

indexed = []
for i, port in enumerate(other_ports, 1):
    indexed.append(
        port.to_indexed2(index=str(i), color="gray", size="medium", overlap=False)
    )

ports_to_draw = [dep_port_indexed, dest_port_indexed, *indexed]

In [7]:
map_image_response, map_image_err =  worker.render_map(coordinates, ports_to_draw, legend_status=False)
with open("image.png", 'bw') as fp:
    fp.write(map_image_response)

In [8]:
map_image_response, map_image_err =  worker.render_map([], ports_to_draw, legend_status=False)
with open("image2.png", 'bw') as fp:
    fp.write(map_image_response)

In [9]:
map_image_response, map_image_err =  worker.render_map(coordinates, [], legend_status=False)
with open("image3.png", 'bw') as fp:
    fp.write(map_image_response)

In [10]:
map_image_err